In [1]:
import os
from pathlib import Path

os.environ.setdefault("MNE_DONTWRITE_HOME", "true")
os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba_cache")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yaml
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from torch.utils.data import DataLoader

from eeg_preprocessing.loaders import CustomDataset, get_subject_ids
from utils.wrappers.cbramod import CBraModWrapper
from utils.wrappers.csbrain import CSBrainWrapper
from utils.wrappers.eegfm import EEGFMWrapper
from utils.wrappers.eegmamba import EEGMambaWrapper
from utils.wrappers.eegnet import EEGNetWrapper
from utils.wrappers.eegpt import EEGPTWrapper
from utils.wrappers.labram import LaBraMWrapper
from utils.wrappers.lumamba import LuMambaWrapper
from utils.wrappers.luna import LUNAWrapper
from utils.wrappers.reve import ReveWrapper

ROOT = Path.cwd()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 100

WRAPPERS = {
    "cbramod": CBraModWrapper,
    "csbrain": CSBrainWrapper,
    "eegfm": EEGFMWrapper,
    "eegmamba": EEGMambaWrapper,
    "eegnet": EEGNetWrapper,
    "eegpt": EEGPTWrapper,
    "labram": LaBraMWrapper,
    "lumamba": LuMambaWrapper,
    "luna": LUNAWrapper,
    "reve": ReveWrapper,
}

CONFIGS = {}
for config_path in sorted((ROOT / "utils" / "configs").glob("*.yaml")):
    with open(config_path) as file:
        CONFIGS[config_path.stem] = yaml.safe_load(file)

LMDB_PATHS = {
    "BCI2A": "/projects/EEG-foundation-model/RECH202/data_preprocessed_normalized/BCI2A/bci2a.lmdb",
    "FACED": "/projects/EEG-foundation-model/RECH202/data_preprocessed_normalized/FACED/faced.lmdb",
    "PHYSIONETMI": "/projects/EEG-foundation-model/RECH202/data_preprocessed_normalized/PHYSIONETMI/physionetmi.lmdb",
}

/home/infres/yrothlin-24/.local/share/mamba/envs/rech202/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class FeatureModel(nn.Module):
    def __init__(self, wrapper):
        super().__init__()
        self.wrapper = wrapper
        self.backbone = wrapper.backbone
        if hasattr(wrapper, "chan_conv"):
            self.chan_conv = wrapper.chan_conv

    def forward(self, x):
        return self.wrapper.forward_features(x)


def create_models(model_names):
    models = {}
    for model_name in model_names:
        if model_name not in WRAPPERS:
            raise ValueError(f"Unknown model: {model_name}")
        config = CONFIGS[model_name]
        checkpoint = None
        if config.get("use_pretrained_weights", False):
            checkpoint_path = ROOT / config["pretrained_weights"]
            if not checkpoint_path.is_file():
                raise FileNotFoundError(checkpoint_path)
            checkpoint = str(checkpoint_path)
        model = WRAPPERS[model_name](checkpoint, dropout=config.get("dropout", 0.0))
        model.model_name = model_name
        models[model_name] = model
    return models


VALID_POOLING_MODES = {"avg_pooling", "last_dim_pooling", "flatten"}


def pool_features(features, pooling_mode="avg_pooling"):
    if pooling_mode not in VALID_POOLING_MODES:
        raise ValueError(
            f"Unknown pooling mode {pooling_mode!r}; expected one of {sorted(VALID_POOLING_MODES)}"
        )
    if features.ndim < 2:
        raise ValueError(f"Expected batched features, got shape {tuple(features.shape)}")

    if pooling_mode == "avg_pooling":
        # Average every intermediate dimension and keep the final embedding dimension.
        dimensions = tuple(range(1, features.ndim - 1))
        return features.mean(dim=dimensions) if dimensions else features
    if pooling_mode == "last_dim_pooling":
        # Average the embedding dimension, then flatten channels/tokens/etc.
        pooled = features.mean(dim=-1)
        return pooled.reshape(pooled.shape[0], -1)
    # Preserve every latent value and flatten all non-batch dimensions.
    return features.reshape(features.shape[0], -1)


@torch.inference_mode()
def load_latent_features(
    model, dataset_name, channel_mode="mapped", pooling_mode="avg_pooling",
    batch_size=64, device=DEVICE,
):
    lmdb_path = LMDB_PATHS[dataset_name]
    subject_ids = get_subject_ids(Path(lmdb_path))
    sample_dataset = CustomDataset(
        lmdb_path,
        subject_ids=[subject_ids[0]],
        channel_mode=channel_mode,
        return_ch_names=channel_mode == "raw",
    )
    sample = sample_dataset[0]
    x_sample = sample[0]
    ch_names = sample[2] if channel_mode == "raw" else None

    if not hasattr(model, "backbone"):
        model.build_backbone(
            n_channels=x_sample.shape[0],
            time_length=x_sample.shape[1],
            ch_names=ch_names,
        )
        model.freeze_backbone()

    feature_model = FeatureModel(model).to(device).eval()
    all_features, all_labels, all_subjects = [], [], []

    for subject_index, subject_id in enumerate(subject_ids):
        dataset = CustomDataset(
            lmdb_path,
            subject_ids=[subject_id],
            channel_mode=channel_mode,
            return_ch_names=channel_mode == "raw",
        )
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
        for batch in loader:
            x, y = batch[:2]
            x = model.preprocess(x)
            x = x.to(device, dtype=torch.float32)
            features = pool_features(feature_model(x), pooling_mode)
            all_features.append(features.float().cpu().numpy())
            all_labels.append(y.numpy())
            all_subjects.append(np.full(len(y), subject_id))
        print(f"{dataset_name}: {subject_index + 1}/{len(subject_ids)} subjects", end="\r")

    print()
    return np.concatenate(all_features), np.concatenate(all_labels), np.concatenate(all_subjects)

In [ ]:
MODEL_NAMES = [
    "reve",
    "cbramod",
    "eegmamba"
    ]
DATASET_NAMES = [
    "BCI2A"
    ]
CHANNEL_MODES = [
    "mapped"
    ]
POOLING_MODES = [
    # "avg_pooling",
    # "last_dim_pooling",
    "flatten"
    ]  

for dataset_name in DATASET_NAMES:
    subject_ids = get_subject_ids(Path(LMDB_PATHS[dataset_name]))
    dataset = CustomDataset(
        LMDB_PATHS[dataset_name],
        subject_ids=subject_ids,
        channel_mode=CHANNEL_MODES[0],
    )
    print(dataset_name, len(dataset), len(subject_ids))

BCI2A 4696 9


In [4]:
def create_linear_classifier(c=1.0):
    return make_pipeline(
        StandardScaler(),
        LinearSVC(C=c, class_weight="balanced", dual="auto", max_iter=20000, random_state=SEED),
    )

In [5]:
def train_linear_probe(
    model_name, dataset_name, channel_mode="mapped",
    pooling_mode="avg_pooling", test_size=0.2,
):
    model = create_models([model_name])[model_name]
    features, labels, subjects = load_latent_features(
        model,
        dataset_name,
        channel_mode=channel_mode,
        pooling_mode=pooling_mode,
    )

    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=SEED)
    train_indices, test_indices = next(splitter.split(features, labels, groups=subjects))
    classifier = create_linear_classifier()
    classifier.fit(features[train_indices], labels[train_indices])
    predictions = classifier.predict(features[test_indices])

    results = pd.Series({
        "model": model_name,
        "dataset": dataset_name,
        "channel_mode": channel_mode,
        "pooling_mode": pooling_mode,
        "train_subjects": np.unique(subjects[train_indices]).size,
        "test_subjects": np.unique(subjects[test_indices]).size,
        "accuracy": accuracy_score(labels[test_indices], predictions),
        "balanced_accuracy": balanced_accuracy_score(labels[test_indices], predictions),
        "f1_macro": f1_score(labels[test_indices], predictions, average="macro"),
    })
    return model, classifier, results


def run_linear_benchmark(model_names, dataset_names, channel_modes, pooling_modes):
    rows, models, classifiers = [], {}, {}
    for dataset_name in dataset_names:
        for model_name in model_names:
            for channel_mode in channel_modes:
                for pooling_mode in pooling_modes:
                    model, classifier, result = train_linear_probe(
                        model_name,
                        dataset_name,
                        channel_mode=channel_mode,
                        pooling_mode=pooling_mode,
                    )
                    key = (model_name, dataset_name, channel_mode, pooling_mode)
                    model.backbone.cpu()
                    if hasattr(model, "chan_conv"):
                        model.chan_conv.cpu()
                    models[key] = model
                    classifiers[key] = classifier
                    rows.append(result)
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
    return pd.DataFrame(rows).reset_index(drop=True), models, classifiers


results, models, classifiers = run_linear_benchmark(
    MODEL_NAMES, DATASET_NAMES, CHANNEL_MODES, POOLING_MODES
)
display(results)

ValueError: Unknown model: revecbramodeegmamba